# How many Airbnb Superhosts are one bad review away?

Airbnb's Superhost programme requires an overall rating of **4.8 or higher**. This notebook takes every
Inside Airbnb city snapshot from June 2026 (123 cities, 37 countries) and asks two questions:

1. **Fragility.** Of the listings whose rating clears 4.8, how many would drop below it after a *single*
   1-star review?
2. **Badge vs rating.** How well does the public rating actually predict who holds the Superhost badge?

It reproduces the published study **exactly** and checks every figure against the released results.

* Write-up: [stellarreply.com/blog/airbnb-superhost-fragility-study.html](https://stellarreply.com/blog/airbnb-superhost-fragility-study.html)
* Dataset DOI: [10.5281/zenodo.22945798](https://doi.org/10.5281/zenodo.22945798)
* Source data: [Inside Airbnb](https://insideairbnb.com/get-the-data/), CC BY 4.0. The input here keeps only four
  fields per listing (review count, rating, Superhost flag, reviews in the last 12 months) and no listing IDs,
  names or host details.

In [ ]:
import glob, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find(name):
    # Kaggle mounts data under /kaggle/input; in the GitHub repo it sits in ../data next to notebook/.
    hits = [h for pattern in (f"/kaggle/input/**/{name}", f"**/{name}", f"../**/{name}")
            for h in glob.glob(pattern, recursive=True) if os.path.isfile(h)]
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

listings = pd.read_csv(find("listing_fields_2026.csv*"), dtype=str, keep_default_na=False)
published_city = pd.read_csv(find("fragility_by_city.csv"))
published_band = pd.read_csv(find("superhost_by_rating_band.csv"))
published_cross = pd.read_csv(find("superhost_by_rating_and_reviews.csv"))
print(f"{len(listings):,} listings in {listings.city.nunique()} cities, {listings.country.nunique()} countries")

## The arithmetic

A displayed rating is an average of whole stars, so for a listing with `n` reviews and rating `r` the total
number of stars is `S = round(r × n)`. Everything else is closed form, not simulation:

* **Buffer** (1-star reviews absorbable while staying at or above 4.8): solve `(S + j) / (n + j) ≥ 4.8`, giving
  `j ≤ (S − 4.8n) / 3.8`.
* **5-star reviews needed** to reach 4.8 from below: solve `(S + 5k) / (n + k) ≥ 4.8`, giving `k ≥ (4.8n − S) / 0.2`.

Listings need at least 10 reviews, Airbnb's own floor for Superhost.

In [ ]:
T, GF, MIN_REVIEWS = 4.8, 4.9, 10

n = pd.to_numeric(listings.number_of_reviews, errors="coerce")
r = pd.to_numeric(listings.review_scores_rating, errors="coerce")
elig = listings[(n >= MIN_REVIEWS) & r.between(1.0, 5.0)].copy()
elig["n"] = n[elig.index].astype(int)
elig["r"] = r[elig.index]

# Star total, clamped to what n reviews can actually sum to. np.round is round-half-even, like Python's round().
elig["S"] = np.clip(np.round(elig.r * elig.n), elig.n, 5 * elig.n).astype(int)
elig["mean"] = elig.S / elig.n
elig["above"] = elig["mean"] >= T
elig["gf"] = elig["mean"] >= GF
elig["buffer"] = np.where(elig.above, np.maximum(0, np.floor((elig.S - T * elig.n) / (T - 1) + 1e-9)), np.nan)
needed = np.ceil((T * elig.n - elig.S) / (5 - T) - 1e-9)
elig["within_5"] = ~elig.above & (needed <= 5)

above = elig[elig.above]
print(f"eligible listings:           {len(elig):,}")
print(f"clear 4.8:                   {len(above):,}  ({100 * len(above) / len(elig):.1f}%)")
print(f"...would drop on one 1-star: {(above.buffer == 0).sum():,}  ({100 * (above.buffer == 0).mean():.1f}% of those above)")
print(f"...would drop within two:    {(above.buffer <= 1).sum():,}  ({100 * (above.buffer <= 1).mean():.1f}%)")
print(f"at 4.9 or above:             {100 * elig.gf.mean():.1f}%")

## Check: does this reproduce the published per-city figures?

Every count for every city is compared with `fragility_by_city.csv`. Any mismatch fails the notebook.

In [ ]:
def city_stats(g):
    a = g[g.above]
    buf = np.sort(np.minimum(a.buffer.to_numpy(), 20))
    return pd.Series({
        "eligible": len(g),
        "above_superhost": len(a),
        "above_guest_favorite": int(g.gf.sum()),
        "buffer_zero": int((a.buffer == 0).sum()),
        "buffer_one_or_less": int((a.buffer <= 1).sum()),
        "median_buffer": int(buf[len(buf) // 2]) if len(buf) else None,
        "within_5_of_superhost": int(g.within_5.sum()),
    })

ours = elig.groupby(["country", "city"]).apply(city_stats).reset_index()
cols = ["eligible", "above_superhost", "above_guest_favorite", "buffer_zero",
        "buffer_one_or_less", "median_buffer", "within_5_of_superhost"]
check = ours.merge(published_city, on=["country", "city"], suffixes=("", "_published"))
assert len(check) == len(published_city) == 123, len(check)
bad = [(row.city, c) for c in cols for row in check.itertuples()
       if getattr(row, c) != getattr(row, c + "_published")]
assert not bad, bad[:10]
print(f"all {len(cols)} figures match for all {len(check)} cities")

## Which markets are most fragile?

Share of listings above 4.8 that would fall below it on one 1-star review, among cities with at least 2,000
eligible listings.

In [ ]:
big = check[check.eligible >= 2000].assign(
    pct_zero_buffer=lambda d: 100 * d.buffer_zero / d.above_superhost,
    pct_above=lambda d: 100 * d.above_superhost / d.eligible)
big.sort_values("pct_zero_buffer", ascending=False)[["city", "country", "eligible", "pct_above", "pct_zero_buffer"]].head(10).round(1)

In [ ]:
print("correlation across these cities between share above 4.8 and share with zero buffer:",
      round(big.pct_above.corr(big.pct_zero_buffer), 2))

High-rating markets are **not** more fragile; if anything, the reverse. What protects a listing is review
volume: the same 4.85 is a thin margin on 20 reviews and a comfortable one on 300.

## Badge vs rating

Now the Superhost flag itself. This uses Inside Airbnb's `host_is_superhost` and the rating as published, and
keeps only listings where the flag is recorded.

In [ ]:
rb = pd.to_numeric(listings.review_scores_rating, errors="coerce")
nb = pd.to_numeric(listings.number_of_reviews, errors="coerce")
flag = listings.host_is_superhost.str.strip()
badge = listings[rb.notna() & nb.notna() & (nb >= MIN_REVIEWS) & flag.isin(["t", "f"])].copy()
badge["r"], badge["n"], badge["sh"] = rb[badge.index], nb[badge.index], flag[badge.index] == "t"

BANDS = [("<4.50", 0, 4.5), ("4.50-4.69", 4.5, 4.7), ("4.70-4.79", 4.7, 4.8),
         ("4.80-4.89", 4.8, 4.9), ("4.90-4.94", 4.9, 4.95), ("4.95+", 4.95, 5.01)]
RBANDS = [("10-24", 10, 25), ("25-49", 25, 50), ("50-99", 50, 100), ("100-249", 100, 250), ("250+", 250, 1e9)]
def band(v, bands):
    for name, lo, hi in bands:
        if lo <= v < hi:
            return name
badge["rating_band"] = badge.r.map(lambda v: band(v, BANDS))
badge["review_band"] = badge.n.map(lambda v: band(v, RBANDS))

by_band = badge.groupby("rating_band").sh.agg(listings="size", superhosts="sum").reindex([b[0] for b in BANDS]).reset_index()
by_band["pct_superhost"] = (100 * by_band.superhosts / by_band.listings).round(1)
assert by_band[["listings", "superhosts"]].values.tolist() == published_band[["listings", "superhosts"]].values.tolist()

cross = badge.groupby(["rating_band", "review_band"]).sh.agg(listings="size", superhosts="sum").reset_index()
merged = cross.merge(published_cross, on=["rating_band", "review_band"], suffixes=("", "_p"))
assert len(merged) == 30 and (merged.listings == merged.listings_p).all() and (merged.superhosts == merged.superhosts_p).all()

below, at_or_above = badge[badge.r < T], badge[badge.r >= T]
print(f"{len(badge):,} listings with a recorded badge status; band and cross-tab figures match the published CSVs")
print(f"below 4.8 that hold Superhost:        {100 * below.sh.mean():.1f}%")
print(f"4.8 or above that do NOT hold it:     {100 * (~at_or_above.sh).mean():.1f}%")
by_band

In [ ]:
ax = by_band.plot.bar(x="rating_band", y="pct_superhost", legend=False, rot=0, figsize=(9, 4.5),
                     color=["#4b5563"] * 3 + ["#16a34a"] * 3)
ax.axvline(2.5, ls="--", color="#b45309")
ax.set_xlabel("All-time rating"); ax.set_ylabel("Share of listings holding Superhost (%)")
ax.set_title("The public rating is a weak guide to the Superhost badge")
for p, v in zip(ax.patches, by_band.pct_superhost):
    ax.annotate(f"{v}%", (p.get_x() + p.get_width() / 2, v + 1), ha="center")
plt.tight_layout(); plt.show()

In [ ]:
pd.crosstab(badge.rating_band, badge.review_band, values=badge.sh, aggfunc="mean") \
  .reindex(index=[b[0] for b in BANDS][::-1], columns=[b[0] for b in RBANDS]).mul(100).round(1)

Read two corners against each other: **below 4.8 with 250+ reviews, about half hold the badge; above it with
10–24 reviews, fewer do.** Volume beats a better rating, and nothing rescues a rating under 4.5.

Why would a well-rated listing not hold the badge? One visible reason is volume in the assessment window.

In [ ]:
badge["ltm"] = pd.to_numeric(listings.loc[badge.index, "number_of_reviews_ltm"], errors="coerce")
ltm = badge.groupby("sh").ltm.median().rename({True: "Superhost", False: "not Superhost"})
print("median reviews in the last 12 months:"); print(ltm.to_string())

## Limitations (read before citing)

* **All-time vs trailing 12 months.** Inside Airbnb publishes the all-time rating; Airbnb assesses Superhost on
  the trailing 12 months, which is not public. That is why the rating and the badge disagree so often.
* **Quarterly lag.** June snapshots sit near the end of an assessment quarter: the badge reflects the 1 April
  assessment, the rating includes everything since.
* **The unit is the listing, not the host.** A host with several listings is counted once per listing.
* **Rating only.** Superhost also needs a 90% response rate, under 1% cancellations and 10 stays.
* **No bookings or revenue.** Inside Airbnb's occupancy estimate is derived from review counts, so any booking
  comparison built on it would be circular.

Licence: CC BY 4.0. Please credit Inside Airbnb and cite the dataset DOI
[10.5281/zenodo.22945798](https://doi.org/10.5281/zenodo.22945798).